# Joint Disambiguation Model Demo

This notebook acts as a quick tour of the attention-based model that improves Gilda's entity grounding by disambiguating all of a source's mentions jointly. The cells below illustrate what the model consists of, how it works, and how it compares to plain Gilda on individual examples and a held-out corpus.

## 1. The problem

Grounding is the process of taking a text mention (e.g. `"ER"`) and assigning it a specific database entity (e.g. `HGNC:3467`). Gilda does this by lexical string matching, returning a ranked list of candidates per mention. Mentions can often be ambiguous, with one mention form being used in different settings to refer to different entities (e.g., `"ER"` can refer to `"estrogen receptor"` or `"endoplasmic reticulum"`). Gilda works around this by optionally consuming the text surrounding a mention, but cannot for input data types that lack this additional text (e.g., a table of mentions). For these data types, the only context available to help resolve ambiguous mentions is other mentions from the same source. Humans often disambiguate this way: for instance, a well-trained scientist who sees the mentions `"breast cancer"`, `"tamoxifen"`, and `"ER"` can infer that `"ER"` means `"estrogen receptor"` in this setting based on the other two and their background knowledge of breast cancer. The model showcased below attempts to mimic this human-like disambiguation strategy with two key tools: (1) a frozen PubMedBERT model to encode the scientific meaning of text spans, and (2) an attention mechanism trained to use those encodings to resolve ambiguous mentions. More concretely, the model re-ranks the gilda-generated candidate lists of all mentions in a source jointly by letting related mentions inform each other.

## 2. How the model works

For each source...
1. A list of candidates is retrieved for every entity mention using Gilda.
2. Every mention and candidate is given an embedding from a pre-trained, frozen PubMedBERT encoder based on their surface forms that represent their semantics.
3. Mentions initialize a "belief" over their candidate list – a probability distribution that gives weight to more likely candidates. For a given mention, this distribution starts proportional to the cosine similarity between mention and candidate embeddings, so that candidates with high semantic similarity are given more starting weight.
4. Three rounds of attention allow mentions to update their beliefs based on source context by attending to other mentions. Since a mention never attends to itself, this is like a mean-field update.
5. The candidate with the most mass of a mention's final belief distribution is taken as its predicted grounding.


The model has a few additional features that contribute to its predictions:

(1) a V-REx penalty is added on top to push the training loss to be equally low across sources grouped by their gold namespace mix. This prevents the model from overfitting sources with certain namespaces and entity types, like chemicals for example, and encourages it to generalize.

(2) A context-only term and orthogonality penalty are added to the loss. The context-only term encourages the model to rely on source context when making a prediction, and the orthogonality penalty encourages what the model learns about source context to be something the mention's text didn't already say.

## 3. Setup

Loads the pre-built corpus and caches straight from disk (no Gilda grounder or BERT needed), then loads the trained model. Runs in a few seconds.

In [1]:
import os
import pickle
import pandas as pd

import ascenda
from ascenda.data import load_corpus, make_splits
from ascenda.train import (load_model, predict_source,
                           DEFAULT_CORPUS_CACHE, DEFAULT_ENTITY_CACHE)
from ascenda.evaluate import filter_ambiguous_mentions, comparison_table

PKG = os.path.dirname(os.path.abspath(ascenda.__file__))
CKPT = os.path.join(PKG, "models", "final_model_best_checkpoint_seed0.pt")

for path in (DEFAULT_CORPUS_CACHE, DEFAULT_ENTITY_CACHE, CKPT):
    assert os.path.exists(path), f"missing artifact: {path}"

datasets = ["bioid", "bc5cdr", "nlmchem", "ncbi_disease", "gnormplus", "medmentions_st21pv"]

sources = load_corpus(datasets, merged_cache=DEFAULT_CORPUS_CACHE)
_, _, test = make_splits(sources)

emb_cache = pickle.load(open(DEFAULT_ENTITY_CACHE, "rb"))

model = load_model(CKPT)
print(f"Loaded {os.path.basename(CKPT)}  |  {len(emb_cache)} entity embeddings  |  {len(test)} test sources")

/Users/bradleybuchner/Desktop/grad_school/research/gyori_lab/ascenda/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded merged corpus (8067 sources) from /Users/bradleybuchner/Desktop/grad_school/research/gyori_lab/ascenda/ascenda/caches/corpus_cache/final_merged.pkl
Loaded final_model_best_checkpoint_seed0.pt  |  22727 entity embeddings  |  1868 test sources


## 4. Gilda vs. Model on a single source

For a single source, view each mention's **Gilda** pick next to what the **model** picks, and whether those picks are correct. The model "fixes" a mention's grounding when its pick is correct and Gilda's is not. It does so by using **other mentions** in the source as context (and importantly, not the source's raw text).

In [2]:
def curie(c):
    return f"{c.term.db}:{c.term.id}"

def compare_source(source):
    preds = predict_source(source, emb_cache, model)
    rows = []
    for m in source.mentions:
        if not m.candidates:
            continue
        gilda_top = m.candidates[0]
        model_top = preds.get(m.text, m.candidates)[0]
        rows.append({
            "mention": m.text,
            "type": m.entity_type,
            "Gilda pick": f"{gilda_top.term.entry_name} ({curie(gilda_top)})",
            "Model pick": f"{model_top.term.entry_name} ({curie(model_top)})",
            "Gilda Correct": (curie(gilda_top) in m.gold_synonyms),
            "Model Correct": (curie(model_top) in m.gold_synonyms),
        })
    return pd.DataFrame(rows)

ambiguous = filter_ambiguous_mentions(test)

def model_gains(source):
    df = compare_source(source)
    return len(df) and (~df["Gilda Correct"] & df["Model Correct"]).any()

examples = [d for d in ambiguous if 2 <= len(d.mentions) <= 6 and model_gains(d)]
print("Source:", examples[10].source_id)
compare_source(examples[10])

Ambiguity filter: kept 6020/68366 mentions (8.8%) across 1004 sources (gap<=0.05, candidates>=2, top>=0.3)


INFO: [2026-08-28 15:32:03] httpx - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: [2026-08-28 15:32:04] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/e1354b7a3a09615f6aba48dfad4b7a613eef7062/config.json "HTTP/1.1 200 OK"
INFO: [2026-08-28 15:32:04] httpx - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: [2026-08-28 15:32:04] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/e1354b7a3a09615f6aba48dfad4b7a613eef7062/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: [2026-08-28 15:32:04] httpx - HTTP Request: GET https://huggingface.co/api/models/microsoft/BiomedNLP-Bi

Source: PMC:4403043


,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,TA,Tissue/Organ,Takayasu Arteritis (MESH:D013625),Eda (UP:O54693),False,False
1,dystrophin,Nonhuman Gene,DMD (HGNC:2928),Dmd (UP:P11531),False,True


## 5. Gilda vs. Model on a real dataset

Here we get a full picture by re-ranking every mention in a corpus of held-out sources with the model and comparing against Gilda, breaking down performance by entity type. The first cell demonstrates this on a full held-out set of sources, and the second on a filtered subset of especially ambiguous mentions from the same corpus.

Comparison table column definitions:

- *Entity Type* – semantic type of the gold entity that a mention refers to
- *Total* – number of mentions across all sources in the corpus
- *Gilda F1* – F1 score attained by grounding with Gilda (no context disambiguation)
- *Model F1* – F1 score attained by re-ranking Gilda candidates with the joint disambiguation model (no context disambiguation)
- *Gains* – Number of previously-incorrect groundings corrected by the joint disambiguation model
- *Losses* – Number of previously-correct groundings made incorrect by the joint disambiguation model
- *Net* – *Gains* minus *Losses*
- *Net %* – *Net* divided by *Total* times 100
- *G/L* – *Gains* divided by *Losses*

The `All` row is the main result: **Model F1** should be greater than **Gilda F1** and **Net** should be greater than 0 if the model corrects more than it breaks. Per type, it helps most where source context is informative (e.g. Disease, Biological Function, Taxon) and is at least better than neutral on others (G/L > 1.0).

In [3]:
print("Full test corpus")
preds = {s.source_id: predict_source(s, emb_cache, model)
         for s in test}
comparison_table(test, preds, model_col="Model F1", total_label="All", add_average_row=False)

Full test corpus


,Entity Type,Total,Gilda F1,Model F1,Gains,Losses,Net,G/L,Net %
0,Biological Function,8113,0.492,0.542,332,11,321,30.18,4.0
1,Cell types/Cell lines,348,0.605,0.615,3,0,3,inf,0.9
2,Cellular Component,356,0.493,0.540,14,0,14,inf,3.9
3,Disease,8976,0.482,0.534,394,21,373,18.76,4.2
4,Human Gene,3384,0.813,0.834,146,78,68,1.87,2.0
5,Nonhuman Gene,859,0.463,0.497,31,4,27,7.75,3.1
6,Small Molecule,24764,0.562,0.672,2570,108,2462,23.80,9.9
7,Taxon,2580,0.531,0.569,79,5,74,15.80,2.9
8,Tissue/Organ,4377,0.527,0.532,24,7,17,3.43,0.4
9,other,14609,0.291,0.296,42,0,42,inf,0.3


In [4]:
print("Ambiguous mentions only")
ambig_preds = {s.source_id: predict_source(s, emb_cache, model)
         for s in ambiguous}
comparison_table(ambiguous, ambig_preds, model_col="Model F1", total_label="All", add_average_row=False)

Ambiguous mentions only


,Entity Type,Total,Gilda F1,Model F1,Gains,Losses,Net,G/L,Net %
0,Biological Function,783,0.386,0.663,239,22,217,10.86,27.7
1,Cell types/Cell lines,28,0.000,0.107,3,0,3,inf,10.7
2,Cellular Component,39,0.667,0.769,4,0,4,inf,10.3
3,Disease,739,0.235,0.629,314,23,291,13.65,39.4
4,Human Gene,302,0.563,0.579,57,52,5,1.10,1.7
5,Nonhuman Gene,85,0.212,0.400,18,2,16,9.00,18.8
6,Small Molecule,3499,0.266,0.526,1020,111,909,9.19,26.0
7,Taxon,120,0.033,0.608,69,0,69,inf,57.5
8,Tissue/Organ,138,0.326,0.391,9,0,9,inf,6.5
9,other,287,0.063,0.178,33,0,33,inf,11.5


## 6. Examples

The cell below mines the held-out corpus in search of example sources where the model fixes Gilda mistakes. Run it with different values of n to see different output sizes.

In [5]:
use_ambiguous = True  # flip to False if you want to look at example sources from the full test set

if use_ambiguous:
    by_id = {s.source_id: s for s in ambiguous}
else:
    by_id = {s.source_id: s for s in test}

def source_flips(source):
    preds = predict_source(source, emb_cache, model)
    gains = []
    losses = []
    seen = set()
    for m in source.mentions:
        if not m.candidates or m.text in seen:
            continue
        seen.add(m.text)
        g = m.candidates[0]
        mo = preds.get(m.text, m.candidates)[0]
        g_ok, mo_ok = curie(g) in m.gold_synonyms, curie(mo) in m.gold_synonyms
        if mo_ok and not g_ok:
            gains.append((m.text, g, mo))
        elif g_ok and not mo_ok:
            losses.append((m.text, g, mo))
    return gains, losses

import random

def mine_examples(sources, n=10, max_mentions=8, seed=0):
    candidates = []
    for d in sources:
        if not (1 <= len(d.mentions) <= max_mentions):
            continue
        gains, losses = source_flips(d)
        if not gains:
            continue
        candidates.append((d.source_id, gains))
    picked = random.Random(seed).sample(candidates, min(n, len(candidates)))
    return {
        source_id: "".join(
            f"  Fixed {t} -> {mo.term.entry_name} ({mo.term.db})"
            f"  [Gilda choice: {g.term.entry_name} ({g.term.db})]\n"
            for t, g, mo in gains[:3])
        for source_id, gains in picked
    }

mined_examples = mine_examples(ambiguous, seed=0)

for source_id, note in mined_examples.items():
    print(f"\n{source_id}: \n{note}")
    df = compare_source(by_id[source_id])
    display(df.drop_duplicates(subset=["mention", "Model pick"]))


PMID:27371369: 
  Fixed multiple sclerosis -> Multiple Sclerosis (MESH)  [Gilda choice: MS (HGNC)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,CER,other,ceramide (CHEBI:CHEBI:17761),ceramide (CHEBI:CHEBI:17761),False,False
3,multiple sclerosis,Biological Function,MS (HGNC:7314),Multiple Sclerosis (MESH:D009103),False,True



PMID:27513357: 
  Fixed mice -> Mice (MESH)  [Gilda choice: MICE (HGNC)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,mice,Taxon,MICE (HGNC:7094),Mice (MESH:D051379),False,True



PMC:4718155: 
  Fixed desmin -> Des (UP)  [Gilda choice: DES (HGNC)]



...

,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,chlorinated hydrocarbons,Small Molecule,chlorohydrocarbon (CHEBI:CHEBI:23115),"Hydrocarbons, Chlorinated (MESH:D006843)",False,True
1,Pb,Small Molecule,lead atom (CHEBI:CHEBI:25016),lead atom (CHEBI:CHEBI:25016),True,True
2,HCl,Small Molecule,hydrogen chloride (CHEBI:CHEBI:17883),hydrogen chloride (CHEBI:CHEBI:17883),True,True
4,amide,Small Molecule,amide (CHEBI:CHEBI:32988),amide (CHEBI:CHEBI:32988),True,True



PMID:27773614: 
  Fixed HCC -> Carcinoma, Hepatocellular (MESH)  [Gilda choice: HYCC1 (HGNC)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,HCC,Biological Function,HYCC1 (HGNC:24587),"Carcinoma, Hepatocellular (MESH:D006528)",False,True



PMID:27297523: 
  Fixed laryngeal cancer -> Laryngeal Neoplasms (MESH)  [Gilda choice: Laryngeal carcinoma (HP)]



,mention,type,Gilda pick,Model pick,Gilda Correct,Model Correct
0,laryngeal cancer,Biological Function,Laryngeal carcinoma (HP:HP:0012118),Laryngeal Neoplasms (MESH:D007822),False,True
2,pathogenesis,Biological Function,cytolysis by symbiont of host cells (GO:GO:000...,modulation by symbiont of host process (GO:GO:...,True,True
